# Appariage stable (Rust) — qui peut tricher dans Gale–Shapley ?

**GameTheory-16i** (sous-famille *matching*) — **approfondissement** du [carnet 16](GameTheory-16-MechanismDesign-Python.ipynb) §5, qui y introduit déjà l'algorithme, la stabilité et l'avantage du côté proposeur.

L'**appariage stable** (*mariage stable*) de **Gale et Shapley (1962)** a valu à **Lloyd Shapley le prix Nobel d'économie 2012**. Le carnet 16 en pose les bases ; ici on pousse la **facette d'incitation** — la question que le §6 ne fait qu'évoquer à travers le théorème de **Gibbard–Satterthwaite** :

> Un agent peut-il améliorer sa situation en **mentant sur ses préférences** ?

Ce carnet implémente l’algorithme, rappelle l’existence et l’optimalité du côté proposant, puis mesure par recherche exhaustive les déclarations sur deux instances 3×3 : aucun homme ne gagne dans C, tandis que w2 améliore son rang en déclarant une autre liste. Ces résultats d’instances ne remplacent pas les théorèmes généraux.

**Objectifs** : exécuter Gale–Shapley, vérifier indépendamment la stabilité, comparer les extrêmes selon le côté proposant et examiner des déclarations stratégiques sur les instances affichées.

**Prérequis** : listes de préférences strictes complètes, dictionnaires et permutations.

**Durée estimée** : 45 minutes.

**Navigation** : [Version Python](GameTheory-16i-Appariage-Stable-Gale-Shapley-Python.ipynb) | [Version C#](GameTheory-16i-Appariage-Stable-Gale-Shapley-CSharp.ipynb) | [Index GameTheory](README.md) | Précédent : [GameTheory-16h — Coopération LLM](GameTheory-16h-Cooperation-LLM-Python.ipynb) | Base : [carnet 16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb).


In [1]:
type Pair = (&'static str, &'static str);
type Prefs = Vec<(&'static str, Vec<&'static str>)>;
type Matching = Vec<(&'static str, &'static str)>;

/// Preferences d'un agent, dans l'ordre declare.
#[allow(dead_code)]
fn prefs_of<'a>(prefs: &'a Prefs, agent: &str) -> &'a Vec<&'static str> {
    prefs.iter().find(|(k, _)| *k == agent).map(|(_, v)| v).expect("agent absent des preferences")
}
/// Cles d'un registre de preferences, dans l'ordre d'insertion.
#[allow(dead_code)]
fn keys(prefs: &Prefs) -> Vec<&'static str> { prefs.iter().map(|(k, _)| *k).collect() }
/// Copie des preferences (les listes sont copiees, comme `copie_prefs`).
#[allow(dead_code)]
fn copie_prefs(prefs: &Prefs) -> Prefs { prefs.iter().map(|(k, v)| (*k, v.clone())).collect() }
/// Rang (0 = premier choix) d'un element dans la liste d'un agent.
#[allow(dead_code)]
fn rank_of(prefs: &Prefs, agent: &str, item: &str) -> usize {
    prefs_of(prefs, agent).iter().position(|x| *x == item).expect("element absent de la liste")
}

/// Partenaire d'une cle dans un appariage (None si absent).
#[allow(dead_code)]
fn match_get(m: &Matching, key: &str) -> Option<&'static str> {
    m.iter().find(|(k, _)| *k == key).map(|(_, v)| *v)
}
/// Ecriture dans un appariage : l'ordre d'insertion est preserve (dict Python).
#[allow(dead_code)]
fn match_set(m: &mut Matching, key: &'static str, value: &'static str) {
    match m.iter_mut().find(|(k, _)| *k == key) {
        Some(entry) => entry.1 = value,
        None => m.push((key, value)),
    }
}
/// Suppression d'une cle (comme `del` sur un dict).
#[allow(dead_code)]
fn match_remove(m: &mut Matching, key: &str) {
    if let Some(i) = m.iter().position(|(k, _)| *k == key) { m.remove(i); }
}
/// Inverse d'un appariage (ordre d'insertion preserve).
#[allow(dead_code)]
fn inverse(m: &Matching) -> Matching {
    let mut out: Matching = Vec::new();
    for (k, v) in m { match_set(&mut out, v, k); }
    out
}

/// Algorithme de Gale-Shapley (recherche differee / deferred acceptance).
///
/// proposers  : les proposeurs, dans l'ordre de traitement (pile LIFO comme `libres.pop()`).
/// prop_prefs : preferences des proposeurs ; resp_prefs : preferences des repondants.
#[allow(dead_code)]
fn gale_shapley(proposers: &[&'static str], prop_prefs: &Prefs, _respondents: &[&'static str], resp_prefs: &Prefs) -> Matching {
    let mut actuel: Matching = Vec::new();     // repondant -> proposeur
    let mut appariage: Matching = Vec::new();  // proposeur  -> repondant
    let mut libres: Vec<&'static str> = proposers.to_vec();
    let mut suivant: Vec<(&'static str, usize)> = proposers.iter().map(|p| (*p, 0)).collect();
    while let Some(p) = libres.pop() {
        let pref = prefs_of(prop_prefs, p).clone();
        let mut i = suivant.iter().find(|(k, _)| *k == p).map(|(_, v)| *v).unwrap_or(0);
        while i < pref.len() {
            let r = pref[i];
            i += 1;
            for e in suivant.iter_mut() { if e.0 == p { e.1 = i; } }
            match match_get(&actuel, r) {
                None => {
                    match_set(&mut actuel, r, p);
                    match_set(&mut appariage, p, r);
                    break;
                }
                Some(q) => {
                    if rank_of(resp_prefs, r, p) < rank_of(resp_prefs, r, q) {
                        match_set(&mut actuel, r, p);
                        match_set(&mut appariage, p, r);
                        match_remove(&mut appariage, q);
                        libres.push(q);
                        break;
                    }
                }
            }
        }
    }
    appariage
}

/// Retourne une paire bloquante (proposeur, respondent), ou None si stable.
#[allow(dead_code)]
fn trouver_paire_bloquante(appariage: &Matching, prop_prefs: &Prefs, resp_prefs: &Prefs) -> Option<Pair> {
    let mut actuel: Matching = Vec::new();
    for (p, r) in appariage { match_set(&mut actuel, r, p); }
    for (p, r) in appariage {
        let pref = prefs_of(prop_prefs, *p).clone();
        for r2 in pref {
            if r2 == *r { break; }                       // p ne prefere personne au-dessus de r
            let a = match_get(&actuel, r2).expect("repondant apparie");
            if rank_of(resp_prefs, r2, *p) < rank_of(resp_prefs, r2, a) {
                return Some((*p, r2));                   // p prefere r2, et r2 prefere p a son actuel
            }
        }
    }
    None
}

// `random.seed(42)` n'alimente aucune sortie de la source : il devient un commentaire.
// Le portage n'utilise que `std`.

println!("Algorithme Gale–Shapley et oracle de paire bloquante définis ; listes strictes complètes attendues.");

Algorithme Gale–Shapley et oracle de paire bloquante définis ; listes strictes complètes attendues.


## 1. Le problème de l'appariage stable

On a **n hommes** et **n femmes**, chacun avec une **liste de préférences** complète et strictement ordonnée sur l'autre sexe. Un **appariage** associe chaque homme à une femme (bijection).

Une **paire bloquante** est une paire (homme, femme) *non* appariés ensemble, où **les deux** préféreraient s'associer l'un à l'autre plutôt qu'à leur partenaire actuel. Un appariage est **stable** s'il n'admet **aucune** paire bloquante.

L'intuition : un appariage instable « s'effondre » — la paire bloquante s'unit et rompt les deux mariages existants. La question est : *existe-t-il toujours un appariage stable, et comment le trouver ?*

In [2]:
type Pair = (&'static str, &'static str);
type Prefs = Vec<(&'static str, Vec<&'static str>)>;
type Matching = Vec<(&'static str, &'static str)>;

/// Preferences d'un agent, dans l'ordre declare.
#[allow(dead_code)]
fn prefs_of<'a>(prefs: &'a Prefs, agent: &str) -> &'a Vec<&'static str> {
    prefs.iter().find(|(k, _)| *k == agent).map(|(_, v)| v).expect("agent absent des preferences")
}
/// Cles d'un registre de preferences, dans l'ordre d'insertion.
#[allow(dead_code)]
fn keys(prefs: &Prefs) -> Vec<&'static str> { prefs.iter().map(|(k, _)| *k).collect() }
/// Copie des preferences (les listes sont copiees, comme `copie_prefs`).
#[allow(dead_code)]
fn copie_prefs(prefs: &Prefs) -> Prefs { prefs.iter().map(|(k, v)| (*k, v.clone())).collect() }
/// Rang (0 = premier choix) d'un element dans la liste d'un agent.
#[allow(dead_code)]
fn rank_of(prefs: &Prefs, agent: &str, item: &str) -> usize {
    prefs_of(prefs, agent).iter().position(|x| *x == item).expect("element absent de la liste")
}

let hommes = ["m1", "m2", "m3"];
let femmes = ["w1", "w2", "w3"];
let h_prefs: Prefs = vec![
    ("m1", vec!["w1", "w2", "w3"]),
    ("m2", vec!["w2", "w3", "w1"]),
    ("m3", vec!["w3", "w1", "w2"]),
];
let f_prefs: Prefs = vec![
    ("w1", vec!["m3", "m1", "m2"]),
    ("w2", vec!["m1", "m2", "m3"]),
    ("w3", vec!["m2", "m3", "m1"]),
];

for &h in hommes.iter() {
    println!("{} : {}", h, prefs_of(&h_prefs, h).join(" > "));
}
println!();
for &f in femmes.iter() {
    println!("{} : {}", f, prefs_of(&f_prefs, f).join(" > "));
};

m1 : w1 > w2 > w3


m2 : w2 > w3 > w1


m3 : w3 > w1 > w2


w1 : m3 > m1 > m2


w2 : m1 > m2 > m3


w3 : m2 > m3 > m1


### Lecture de l'instance A

La sortie ci-dessus expose la structure du conflit : chaque homme a un premier choix **distinct** (`m1→w1`, `m2→w2`, `m3→w3`) — pas de compétition apparente — et chaque femme aussi (`w1→m3`, `w2→m1`, `w3→m2`). Mais ces deux « mondes idéaux » ne coïncident pas : `w1` veut `m3`, qui préfère `w3` ; `w2` veut `m1`, qui préfère `w1`. Personne n'est partagé, et pourtant aucun accord évident n'émerge — c'est exactement la situation où le critère réaliste n'est plus l'unanimité mais la **stabilité** : un appariement qu'aucun couple ne veut quitter ensemble.

Concrètement, cette instance admet **deux** appariements stables, et c'est ce qui rend l'exemple instructif : le premier, `{m1: w1, m2: w2, m3: w3}`, et le second, `{m1: w2, m2: w3, m3: w1}`. Vérifions pourquoi chacun est stable : on cherche une **paire bloquante**, c'est-à-dire un homme et une femme qui s'aiment mutuellement plus que leur partenaire actuel. Dans le premier, `w1` aimerait `m3`, mais `m3` est engagé à `w3` qu'il préfère — le couple `(m3, w1)` n'est donc pas bloquant, car un des deux (en l'occurrence `m3`) n'a aucune envie de fuir ; le même raisonnement élimine chaque autre frustration. Le raisonnement tient symétriquement dans le second appariement. Deux structures stables coexistent et aucune n'est « la » solution : le choix entre elles dépendra de **qui propose**, comme le mesurera la section suivante. C'est ce qui distingue l'appariage stable d'un problème d'optimisation à solution unique.


## 2. L'algorithme de Gale–Shapley (*deferred acceptance*)

L'algorithme fait **proposer** à un côté (ici les hommes) :

1. un homme **libre** propose à la **première** femme de sa liste qu'il n'a pas encore sollicitée ;
2. si la femme est **libre**, elle accepte temporairement ;
3. si elle est **engagée**, elle garde le **préféré** entre son prétendant actuel et le nouveau, et libère l'autre ;
4. on répète jusqu'à ce que plus personne ne soit libre.

Chaque engagement est **réversible** (d'où *deferred acceptance*) : une femme peut « lâcher » son partenaire actuel pour un meilleur.

Pour ancrer le mécanisme, suivons les premières itérations sur l'instance A. Au départ, les trois hommes sont libres et proposent chacun à son **premier** choix : `m1→w1`, `m2→w2`, `m3→w3`. Les trois femmes sont libres, elles acceptent chacune, et plus d'homme n'est libre — l'algorithme **s'arrête immédiatement**, avec l'appariement `{m1: w1, m2: w2, m3: w3}`. Sur cette instance il n'y a donc aucune itération de « lâcher » : les premiers choix des hommes sont distincts ; les premiers choix des femmes le sont également. C'est une lecture utile du résultat 1, mais ne la sur-généralisons pas : d'autres instances (la C, plus bas) font **tourner** la boucle, avec des engagements réversibles en chaîne, jusqu'à l'impasse. C'est le *deferred acceptance* qui rend ce parcours toujours bien défini : un engagement tenu pour l'instant, révocable dès qu'un meilleur prétendant arrive.

L'algorithme **termine toujours** (chaque proposition élimine définitivement une paire homme-femme, et il en est un nombre fini) et produit un appariage **stable**.



### Exercice 1 — Reconstruire l’acceptation différée
<a id="exercice-1"></a>

Implémentez la boucle de propositions avec la pile LIFO et les préférences affichées. Le code précédent fournit un modèle complet à étudier.

In [3]:
type Pair = (&'static str, &'static str);
type Prefs = Vec<(&'static str, Vec<&'static str>)>;
type Matching = Vec<(&'static str, &'static str)>;

/// Preferences d'un agent, dans l'ordre declare.
#[allow(dead_code)]
fn prefs_of<'a>(prefs: &'a Prefs, agent: &str) -> &'a Vec<&'static str> {
    prefs.iter().find(|(k, _)| *k == agent).map(|(_, v)| v).expect("agent absent des preferences")
}
/// Cles d'un registre de preferences, dans l'ordre d'insertion.
#[allow(dead_code)]
fn keys(prefs: &Prefs) -> Vec<&'static str> { prefs.iter().map(|(k, _)| *k).collect() }
/// Copie des preferences (les listes sont copiees, comme `copie_prefs`).
#[allow(dead_code)]
fn copie_prefs(prefs: &Prefs) -> Prefs { prefs.iter().map(|(k, v)| (*k, v.clone())).collect() }
/// Rang (0 = premier choix) d'un element dans la liste d'un agent.
#[allow(dead_code)]
fn rank_of(prefs: &Prefs, agent: &str, item: &str) -> usize {
    prefs_of(prefs, agent).iter().position(|x| *x == item).expect("element absent de la liste")
}

/// Partenaire d'une cle dans un appariage (None si absent).
#[allow(dead_code)]
fn match_get(m: &Matching, key: &str) -> Option<&'static str> {
    m.iter().find(|(k, _)| *k == key).map(|(_, v)| *v)
}
/// Ecriture dans un appariage : l'ordre d'insertion est preserve (dict Python).
#[allow(dead_code)]
fn match_set(m: &mut Matching, key: &'static str, value: &'static str) {
    match m.iter_mut().find(|(k, _)| *k == key) {
        Some(entry) => entry.1 = value,
        None => m.push((key, value)),
    }
}
/// Suppression d'une cle (comme `del` sur un dict).
#[allow(dead_code)]
fn match_remove(m: &mut Matching, key: &str) {
    if let Some(i) = m.iter().position(|(k, _)| *k == key) { m.remove(i); }
}
/// Inverse d'un appariage (ordre d'insertion preserve).
#[allow(dead_code)]
fn inverse(m: &Matching) -> Matching {
    let mut out: Matching = Vec::new();
    for (k, v) in m { match_set(&mut out, v, k); }
    out
}

/// Repr Python d'un appariage (dict) : {'m3': 'w3', ...}
#[allow(dead_code)]
fn py_dict_repr(m: &Matching) -> String {
    let items: Vec<String> = m.iter().map(|(k, v)| format!("'{}': '{}'", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}
/// Repr Python d'une liste de chaines : ['m3', 'm1', 'm2']
#[allow(dead_code)]
fn py_list_repr(v: &[&'static str]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
/// Repr Python d'un booleen.
#[allow(dead_code)]
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
/// Repr Python d'un couple (proposeur, respondent).
#[allow(dead_code)]
fn py_pair_repr(p: Option<Pair>) -> String {
    match p {
        None => "None".to_string(),
        Some((a, b)) => format!("('{}', '{}')", a, b),
    }
}
/// Egalite de dictionnaires : insensible a l'ordre des cles.
#[allow(dead_code)]
fn dict_eq(a: &Matching, b: &Matching) -> bool {
    a.len() == b.len() && a.iter().all(|(k, v)| match_get(b, k) == Some(*v))
}

/// Algorithme de Gale-Shapley (recherche differee / deferred acceptance).
///
/// proposers  : les proposeurs, dans l'ordre de traitement (pile LIFO comme `libres.pop()`).
/// prop_prefs : preferences des proposeurs ; resp_prefs : preferences des repondants.
#[allow(dead_code)]
fn gale_shapley(proposers: &[&'static str], prop_prefs: &Prefs, _respondents: &[&'static str], resp_prefs: &Prefs) -> Matching {
    let mut actuel: Matching = Vec::new();     // repondant -> proposeur
    let mut appariage: Matching = Vec::new();  // proposeur  -> repondant
    let mut libres: Vec<&'static str> = proposers.to_vec();
    let mut suivant: Vec<(&'static str, usize)> = proposers.iter().map(|p| (*p, 0)).collect();
    while let Some(p) = libres.pop() {
        let pref = prefs_of(prop_prefs, p).clone();
        let mut i = suivant.iter().find(|(k, _)| *k == p).map(|(_, v)| *v).unwrap_or(0);
        while i < pref.len() {
            let r = pref[i];
            i += 1;
            for e in suivant.iter_mut() { if e.0 == p { e.1 = i; } }
            match match_get(&actuel, r) {
                None => {
                    match_set(&mut actuel, r, p);
                    match_set(&mut appariage, p, r);
                    break;
                }
                Some(q) => {
                    if rank_of(resp_prefs, r, p) < rank_of(resp_prefs, r, q) {
                        match_set(&mut actuel, r, p);
                        match_set(&mut appariage, p, r);
                        match_remove(&mut appariage, q);
                        libres.push(q);
                        break;
                    }
                }
            }
        }
    }
    appariage
}

/// Retourne une paire bloquante (proposeur, respondent), ou None si stable.
#[allow(dead_code)]
fn trouver_paire_bloquante(appariage: &Matching, prop_prefs: &Prefs, resp_prefs: &Prefs) -> Option<Pair> {
    let mut actuel: Matching = Vec::new();
    for (p, r) in appariage { match_set(&mut actuel, r, p); }
    for (p, r) in appariage {
        let pref = prefs_of(prop_prefs, *p).clone();
        for r2 in pref {
            if r2 == *r { break; }                       // p ne prefere personne au-dessus de r
            let a = match_get(&actuel, r2).expect("repondant apparie");
            if rank_of(resp_prefs, r2, *p) < rank_of(resp_prefs, r2, a) {
                return Some((*p, r2));                   // p prefere r2, et r2 prefere p a son actuel
            }
        }
    }
    None
}

let hommes = ["m1", "m2", "m3"];
let femmes = ["w1", "w2", "w3"];
let h_prefs: Prefs = vec![
    ("m1", vec!["w1", "w2", "w3"]),
    ("m2", vec!["w2", "w3", "w1"]),
    ("m3", vec!["w3", "w1", "w2"]),
];
let f_prefs: Prefs = vec![
    ("w1", vec!["m3", "m1", "m2"]),
    ("w2", vec!["m1", "m2", "m3"]),
    ("w3", vec!["m2", "m3", "m1"]),
];

// Exercice 1 : la boucle de recherche differee reste a ecrire.
fn gale_shapley_etudiant(_proposers: &[&'static str], _prop_prefs: &Prefs, _respondents: &[&'static str], _resp_prefs: &Prefs) -> Option<Matching> {
    // TODO etudiant : retourner l'appariage {homme: femme}
    None
}

let deux_p: Prefs = vec![("p1", vec!["r1", "r2"]), ("p2", vec!["r1", "r2"])];
let deux_r: Prefs = vec![("r1", vec!["p1", "p2"]), ("r2", vec!["p2", "p1"])];

let fixtures: Vec<(Prefs, Prefs)> = vec![
    (h_prefs.clone(), f_prefs.clone()),
    (deux_p.clone(), deux_r.clone()),
    (vec![("p", vec!["r"])], vec![("r", vec!["p"])]),
    (Vec::new(), Vec::new()),
];
let mut reponse: Option<Matching> = None;
let mut termine = true;
'fixtures: for (pp, rp) in fixtures.iter() {
    let cles = keys(pp);
    let cles_inversees: Vec<&'static str> = cles.iter().rev().cloned().collect();
    for ordre in [cles.clone(), cles_inversees].iter() {
        let attendu = gale_shapley(ordre, pp, &keys(rp), rp);
        reponse = gale_shapley_etudiant(ordre, &copie_prefs(pp), &keys(rp), &copie_prefs(rp));
        match &reponse {
            None => break,
            Some(r) => { assert!(dict_eq(r, &attendu), "Ex1 appariage incorrect"); }
        }
    }
    if reponse.is_none() {
        println!("Exercice 1 à compléter : A, remplacements, singleton et vide");
        termine = false;
        break 'fixtures;
    }
}
if termine {
    println!("Exercice 1 : quatre fixtures et ordres inversés validés");
};

Exercice 1 à compléter : A, remplacements, singleton et vide


### Exécuter le résultat de référence

Les réponses étudiantes restent ouvertes. La cellule suivante calcule le résultat du modèle de référence sur les préférences affichées.

In [4]:
type Pair = (&'static str, &'static str);
type Prefs = Vec<(&'static str, Vec<&'static str>)>;
type Matching = Vec<(&'static str, &'static str)>;

/// Preferences d'un agent, dans l'ordre declare.
#[allow(dead_code)]
fn prefs_of<'a>(prefs: &'a Prefs, agent: &str) -> &'a Vec<&'static str> {
    prefs.iter().find(|(k, _)| *k == agent).map(|(_, v)| v).expect("agent absent des preferences")
}
/// Cles d'un registre de preferences, dans l'ordre d'insertion.
#[allow(dead_code)]
fn keys(prefs: &Prefs) -> Vec<&'static str> { prefs.iter().map(|(k, _)| *k).collect() }
/// Copie des preferences (les listes sont copiees, comme `copie_prefs`).
#[allow(dead_code)]
fn copie_prefs(prefs: &Prefs) -> Prefs { prefs.iter().map(|(k, v)| (*k, v.clone())).collect() }
/// Rang (0 = premier choix) d'un element dans la liste d'un agent.
#[allow(dead_code)]
fn rank_of(prefs: &Prefs, agent: &str, item: &str) -> usize {
    prefs_of(prefs, agent).iter().position(|x| *x == item).expect("element absent de la liste")
}

/// Partenaire d'une cle dans un appariage (None si absent).
#[allow(dead_code)]
fn match_get(m: &Matching, key: &str) -> Option<&'static str> {
    m.iter().find(|(k, _)| *k == key).map(|(_, v)| *v)
}
/// Ecriture dans un appariage : l'ordre d'insertion est preserve (dict Python).
#[allow(dead_code)]
fn match_set(m: &mut Matching, key: &'static str, value: &'static str) {
    match m.iter_mut().find(|(k, _)| *k == key) {
        Some(entry) => entry.1 = value,
        None => m.push((key, value)),
    }
}
/// Suppression d'une cle (comme `del` sur un dict).
#[allow(dead_code)]
fn match_remove(m: &mut Matching, key: &str) {
    if let Some(i) = m.iter().position(|(k, _)| *k == key) { m.remove(i); }
}
/// Inverse d'un appariage (ordre d'insertion preserve).
#[allow(dead_code)]
fn inverse(m: &Matching) -> Matching {
    let mut out: Matching = Vec::new();
    for (k, v) in m { match_set(&mut out, v, k); }
    out
}

/// Repr Python d'un appariage (dict) : {'m3': 'w3', ...}
#[allow(dead_code)]
fn py_dict_repr(m: &Matching) -> String {
    let items: Vec<String> = m.iter().map(|(k, v)| format!("'{}': '{}'", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}
/// Repr Python d'une liste de chaines : ['m3', 'm1', 'm2']
#[allow(dead_code)]
fn py_list_repr(v: &[&'static str]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
/// Repr Python d'un booleen.
#[allow(dead_code)]
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
/// Repr Python d'un couple (proposeur, respondent).
#[allow(dead_code)]
fn py_pair_repr(p: Option<Pair>) -> String {
    match p {
        None => "None".to_string(),
        Some((a, b)) => format!("('{}', '{}')", a, b),
    }
}
/// Egalite de dictionnaires : insensible a l'ordre des cles.
#[allow(dead_code)]
fn dict_eq(a: &Matching, b: &Matching) -> bool {
    a.len() == b.len() && a.iter().all(|(k, v)| match_get(b, k) == Some(*v))
}

/// Algorithme de Gale-Shapley (recherche differee / deferred acceptance).
///
/// proposers  : les proposeurs, dans l'ordre de traitement (pile LIFO comme `libres.pop()`).
/// prop_prefs : preferences des proposeurs ; resp_prefs : preferences des repondants.
#[allow(dead_code)]
fn gale_shapley(proposers: &[&'static str], prop_prefs: &Prefs, _respondents: &[&'static str], resp_prefs: &Prefs) -> Matching {
    let mut actuel: Matching = Vec::new();     // repondant -> proposeur
    let mut appariage: Matching = Vec::new();  // proposeur  -> repondant
    let mut libres: Vec<&'static str> = proposers.to_vec();
    let mut suivant: Vec<(&'static str, usize)> = proposers.iter().map(|p| (*p, 0)).collect();
    while let Some(p) = libres.pop() {
        let pref = prefs_of(prop_prefs, p).clone();
        let mut i = suivant.iter().find(|(k, _)| *k == p).map(|(_, v)| *v).unwrap_or(0);
        while i < pref.len() {
            let r = pref[i];
            i += 1;
            for e in suivant.iter_mut() { if e.0 == p { e.1 = i; } }
            match match_get(&actuel, r) {
                None => {
                    match_set(&mut actuel, r, p);
                    match_set(&mut appariage, p, r);
                    break;
                }
                Some(q) => {
                    if rank_of(resp_prefs, r, p) < rank_of(resp_prefs, r, q) {
                        match_set(&mut actuel, r, p);
                        match_set(&mut appariage, p, r);
                        match_remove(&mut appariage, q);
                        libres.push(q);
                        break;
                    }
                }
            }
        }
    }
    appariage
}

/// Retourne une paire bloquante (proposeur, respondent), ou None si stable.
#[allow(dead_code)]
fn trouver_paire_bloquante(appariage: &Matching, prop_prefs: &Prefs, resp_prefs: &Prefs) -> Option<Pair> {
    let mut actuel: Matching = Vec::new();
    for (p, r) in appariage { match_set(&mut actuel, r, p); }
    for (p, r) in appariage {
        let pref = prefs_of(prop_prefs, *p).clone();
        for r2 in pref {
            if r2 == *r { break; }                       // p ne prefere personne au-dessus de r
            let a = match_get(&actuel, r2).expect("repondant apparie");
            if rank_of(resp_prefs, r2, *p) < rank_of(resp_prefs, r2, a) {
                return Some((*p, r2));                   // p prefere r2, et r2 prefere p a son actuel
            }
        }
    }
    None
}

let hommes = ["m1", "m2", "m3"];
let femmes = ["w1", "w2", "w3"];
let h_prefs: Prefs = vec![
    ("m1", vec!["w1", "w2", "w3"]),
    ("m2", vec!["w2", "w3", "w1"]),
    ("m3", vec!["w3", "w1", "w2"]),
];
let f_prefs: Prefs = vec![
    ("w1", vec!["m3", "m1", "m2"]),
    ("w2", vec!["m1", "m2", "m3"]),
    ("w3", vec!["m2", "m3", "m1"]),
];

let appariage = gale_shapley(&hommes, &h_prefs, &femmes, &f_prefs);
println!("Appariage produit (hommes proposent) : {}", py_dict_repr(&appariage));

let pb = trouver_paire_bloquante(&appariage, &h_prefs, &f_prefs);
let verdict = if pb.is_none() { "  ->  stable" } else { "  ->  INSTABLE" };
println!("Paire bloquante : {} {}", py_pair_repr(pb), verdict);

Appariage produit (hommes proposent) : {'m3': 'w3', 'm2': 'w2', 'm1': 'w1'}


Paire bloquante : None   ->  stable


### Résultat 1 — Existence

La mesure le confirme : l'algorithme **termine** et l'appariement produit — ici `{m1: w1, m2: w2, m3: w3}` — ne contient **aucune paire bloquante** (`trouver_paire_bloquante` retourne `None`).

Un détail de la sortie mérite une lecture : chaque homme obtient son **premier choix**, et pourtant l'appariement n'est pas « idéal pour tous ». `w1` aurait préféré `m3` — mais `m3` a obtenu `w3`, *son* premier choix, donc le couple `(m3, w1)` n'est pas bloquant : un seul des deux a envie de fuir. **Stable ≠ unanime** : la stabilité n'exige l'absence que des paires *mutuellement* insatisfaites.

Pourquoi la termination est-elle si forte ici ? Chaque fois qu'un homme est rejeté, il ne **reproposera jamais** à cette femme : sa position dans sa propre liste ne fait que **descendre**, et une liste est finie. Le nombre total de propositions possibles est donc borné par `n²`, et aucune boucle ne peut tourner indéfiniment. La termination n'est pas une heuristique : c'est elle qui **prouve** l'existence, car à l'instant où l'algorithme s'arrête, il tient un appariement sans paire bloquante.

C'est le théorème d'**existence** de Gale–Shapley : *pour des listes strictes complètes de même taille, il existe au moins un appariement stable* — et l'algorithme le trouve en **temps polynomial** (O(n²) propositions au plus).



### Exercice 2 — Vérifier une paire bloquante
<a id="exercice-2"></a>

Écrivez un oracle indépendant de Gale–Shapley : il doit détecter une paire qui se préfère mutuellement et renvoyer `None` seulement si le matching est stable.

In [5]:
type Pair = (&'static str, &'static str);
type Prefs = Vec<(&'static str, Vec<&'static str>)>;
type Matching = Vec<(&'static str, &'static str)>;

/// Preferences d'un agent, dans l'ordre declare.
#[allow(dead_code)]
fn prefs_of<'a>(prefs: &'a Prefs, agent: &str) -> &'a Vec<&'static str> {
    prefs.iter().find(|(k, _)| *k == agent).map(|(_, v)| v).expect("agent absent des preferences")
}
/// Cles d'un registre de preferences, dans l'ordre d'insertion.
#[allow(dead_code)]
fn keys(prefs: &Prefs) -> Vec<&'static str> { prefs.iter().map(|(k, _)| *k).collect() }
/// Copie des preferences (les listes sont copiees, comme `copie_prefs`).
#[allow(dead_code)]
fn copie_prefs(prefs: &Prefs) -> Prefs { prefs.iter().map(|(k, v)| (*k, v.clone())).collect() }
/// Rang (0 = premier choix) d'un element dans la liste d'un agent.
#[allow(dead_code)]
fn rank_of(prefs: &Prefs, agent: &str, item: &str) -> usize {
    prefs_of(prefs, agent).iter().position(|x| *x == item).expect("element absent de la liste")
}

/// Partenaire d'une cle dans un appariage (None si absent).
#[allow(dead_code)]
fn match_get(m: &Matching, key: &str) -> Option<&'static str> {
    m.iter().find(|(k, _)| *k == key).map(|(_, v)| *v)
}
/// Ecriture dans un appariage : l'ordre d'insertion est preserve (dict Python).
#[allow(dead_code)]
fn match_set(m: &mut Matching, key: &'static str, value: &'static str) {
    match m.iter_mut().find(|(k, _)| *k == key) {
        Some(entry) => entry.1 = value,
        None => m.push((key, value)),
    }
}
/// Suppression d'une cle (comme `del` sur un dict).
#[allow(dead_code)]
fn match_remove(m: &mut Matching, key: &str) {
    if let Some(i) = m.iter().position(|(k, _)| *k == key) { m.remove(i); }
}
/// Inverse d'un appariage (ordre d'insertion preserve).
#[allow(dead_code)]
fn inverse(m: &Matching) -> Matching {
    let mut out: Matching = Vec::new();
    for (k, v) in m { match_set(&mut out, v, k); }
    out
}

/// Repr Python d'un appariage (dict) : {'m3': 'w3', ...}
#[allow(dead_code)]
fn py_dict_repr(m: &Matching) -> String {
    let items: Vec<String> = m.iter().map(|(k, v)| format!("'{}': '{}'", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}
/// Repr Python d'une liste de chaines : ['m3', 'm1', 'm2']
#[allow(dead_code)]
fn py_list_repr(v: &[&'static str]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
/// Repr Python d'un booleen.
#[allow(dead_code)]
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
/// Repr Python d'un couple (proposeur, respondent).
#[allow(dead_code)]
fn py_pair_repr(p: Option<Pair>) -> String {
    match p {
        None => "None".to_string(),
        Some((a, b)) => format!("('{}', '{}')", a, b),
    }
}
/// Egalite de dictionnaires : insensible a l'ordre des cles.
#[allow(dead_code)]
fn dict_eq(a: &Matching, b: &Matching) -> bool {
    a.len() == b.len() && a.iter().all(|(k, v)| match_get(b, k) == Some(*v))
}

/// Permutations dans l'ordre de `itertools.permutations` (ordre lexicographique des positions).
#[allow(dead_code)]
fn permutations<T: Clone>(items: &[T]) -> Vec<Vec<T>> {
    let n = items.len();
    let mut out: Vec<Vec<T>> = Vec::new();
    if n == 0 { out.push(Vec::new()); return out; }
    let mut idx: Vec<usize> = (0..n).collect();
    loop {
        out.push(idx.iter().map(|i| items[*i].clone()).collect());
        let mut i = n - 1;
        while i > 0 && idx[i - 1] >= idx[i] { i -= 1; }
        if i == 0 { break; }
        let mut j = n - 1;
        while idx[j] <= idx[i - 1] { j -= 1; }
        idx.swap(i - 1, j);
        idx[i..].reverse();
    }
    out
}

/// Algorithme de Gale-Shapley (recherche differee / deferred acceptance).
///
/// proposers  : les proposeurs, dans l'ordre de traitement (pile LIFO comme `libres.pop()`).
/// prop_prefs : preferences des proposeurs ; resp_prefs : preferences des repondants.
#[allow(dead_code)]
fn gale_shapley(proposers: &[&'static str], prop_prefs: &Prefs, _respondents: &[&'static str], resp_prefs: &Prefs) -> Matching {
    let mut actuel: Matching = Vec::new();     // repondant -> proposeur
    let mut appariage: Matching = Vec::new();  // proposeur  -> repondant
    let mut libres: Vec<&'static str> = proposers.to_vec();
    let mut suivant: Vec<(&'static str, usize)> = proposers.iter().map(|p| (*p, 0)).collect();
    while let Some(p) = libres.pop() {
        let pref = prefs_of(prop_prefs, p).clone();
        let mut i = suivant.iter().find(|(k, _)| *k == p).map(|(_, v)| *v).unwrap_or(0);
        while i < pref.len() {
            let r = pref[i];
            i += 1;
            for e in suivant.iter_mut() { if e.0 == p { e.1 = i; } }
            match match_get(&actuel, r) {
                None => {
                    match_set(&mut actuel, r, p);
                    match_set(&mut appariage, p, r);
                    break;
                }
                Some(q) => {
                    if rank_of(resp_prefs, r, p) < rank_of(resp_prefs, r, q) {
                        match_set(&mut actuel, r, p);
                        match_set(&mut appariage, p, r);
                        match_remove(&mut appariage, q);
                        libres.push(q);
                        break;
                    }
                }
            }
        }
    }
    appariage
}

/// Retourne une paire bloquante (proposeur, respondent), ou None si stable.
#[allow(dead_code)]
fn trouver_paire_bloquante(appariage: &Matching, prop_prefs: &Prefs, resp_prefs: &Prefs) -> Option<Pair> {
    let mut actuel: Matching = Vec::new();
    for (p, r) in appariage { match_set(&mut actuel, r, p); }
    for (p, r) in appariage {
        let pref = prefs_of(prop_prefs, *p).clone();
        for r2 in pref {
            if r2 == *r { break; }                       // p ne prefere personne au-dessus de r
            let a = match_get(&actuel, r2).expect("repondant apparie");
            if rank_of(resp_prefs, r2, *p) < rank_of(resp_prefs, r2, a) {
                return Some((*p, r2));                   // p prefere r2, et r2 prefere p a son actuel
            }
        }
    }
    None
}

let hommes = ["m1", "m2", "m3"];
let femmes = ["w1", "w2", "w3"];
let h_prefs: Prefs = vec![
    ("m1", vec!["w1", "w2", "w3"]),
    ("m2", vec!["w2", "w3", "w1"]),
    ("m3", vec!["w3", "w1", "w2"]),
];
let f_prefs: Prefs = vec![
    ("w1", vec!["m3", "m1", "m2"]),
    ("w2", vec!["m1", "m2", "m3"]),
    ("w3", vec!["m2", "m3", "m1"]),
];

// Exercice 2 : le verificateur de stabilite reste a ecrire.
fn trouver_paire_bloquante_etudiant(_appariage: &Matching, _homes_prefs: &Prefs, _femmes_prefs: &Prefs) -> Option<Pair> {
    // TODO etudiant : la paire bloquante (homme, femme), ou None si stable
    None
}

/// Controle independant : la paire retournee est-elle reellement bloquante ?
#[allow(dead_code)]
fn paire_effectivement_bloquante(paire: Option<Pair>, appariage: &Matching, pp: &Prefs, rp: &Prefs) -> bool {
    let (p, r) = match paire { Some(v) => v, None => return false };
    if !keys(pp).contains(&p) || !keys(rp).contains(&r) { return false; }
    let partenaire_p = match_get(appariage, p).expect("proposeur apparie");
    if partenaire_p == r { return false; }
    let miroir = inverse(appariage);
    let partenaire_r = match_get(&miroir, r).expect("repondant apparie");
    rank_of(pp, p, r) < rank_of(pp, p, partenaire_p)
        && rank_of(rp, r, p) < rank_of(rp, r, partenaire_r)
}

let deux_p: Prefs = vec![("p1", vec!["r1", "r2"]), ("p2", vec!["r1", "r2"])];
let deux_r: Prefs = vec![("r1", vec!["p1", "p2"]), ("r2", vec!["p2", "p1"])];

let verificateur_etudiant: Option<fn(&Matching, &Prefs, &Prefs) -> Option<Pair>> = None;
match verificateur_etudiant {
    None => println!("Exercice 2 à compléter : fonction et callback de vérification"),
    Some(verificateur) => {
        let mut controles = 0usize;
        for (pp, rp) in [(h_prefs.clone(), f_prefs.clone()), (deux_p.clone(), deux_r.clone())].iter() {
            for ordre in permutations(&keys(rp)) {
                let mut appariage: Matching = Vec::new();
                for (k, v) in keys(pp).iter().zip(ordre.iter()) { match_set(&mut appariage, k, v); }
                let stable = trouver_paire_bloquante(&appariage, pp, rp).is_none();
                let reponse = verificateur(&appariage, &copie_prefs(pp), &copie_prefs(rp));
                if stable {
                    assert!(reponse.is_none(), "Ex2 paire non bloquante ou absence incorrecte");
                } else {
                    assert!(paire_effectivement_bloquante(reponse, &appariage, pp, rp),
                            "Ex2 paire non bloquante ou absence incorrecte");
                }
                controles += 1;
            }
        }
        println!("Exercice 2 : {} bijections et paires effectivement bloquantes contrôlées", controles);
    }
};

Exercice 2 à compléter : fonction et callback de vérification


### Exécuter le résultat de référence

Les réponses étudiantes restent ouvertes. La cellule suivante calcule le résultat du modèle de référence sur les préférences affichées.

In [6]:
type Pair = (&'static str, &'static str);
type Prefs = Vec<(&'static str, Vec<&'static str>)>;
type Matching = Vec<(&'static str, &'static str)>;

/// Preferences d'un agent, dans l'ordre declare.
#[allow(dead_code)]
fn prefs_of<'a>(prefs: &'a Prefs, agent: &str) -> &'a Vec<&'static str> {
    prefs.iter().find(|(k, _)| *k == agent).map(|(_, v)| v).expect("agent absent des preferences")
}
/// Cles d'un registre de preferences, dans l'ordre d'insertion.
#[allow(dead_code)]
fn keys(prefs: &Prefs) -> Vec<&'static str> { prefs.iter().map(|(k, _)| *k).collect() }
/// Copie des preferences (les listes sont copiees, comme `copie_prefs`).
#[allow(dead_code)]
fn copie_prefs(prefs: &Prefs) -> Prefs { prefs.iter().map(|(k, v)| (*k, v.clone())).collect() }
/// Rang (0 = premier choix) d'un element dans la liste d'un agent.
#[allow(dead_code)]
fn rank_of(prefs: &Prefs, agent: &str, item: &str) -> usize {
    prefs_of(prefs, agent).iter().position(|x| *x == item).expect("element absent de la liste")
}

/// Partenaire d'une cle dans un appariage (None si absent).
#[allow(dead_code)]
fn match_get(m: &Matching, key: &str) -> Option<&'static str> {
    m.iter().find(|(k, _)| *k == key).map(|(_, v)| *v)
}
/// Ecriture dans un appariage : l'ordre d'insertion est preserve (dict Python).
#[allow(dead_code)]
fn match_set(m: &mut Matching, key: &'static str, value: &'static str) {
    match m.iter_mut().find(|(k, _)| *k == key) {
        Some(entry) => entry.1 = value,
        None => m.push((key, value)),
    }
}
/// Suppression d'une cle (comme `del` sur un dict).
#[allow(dead_code)]
fn match_remove(m: &mut Matching, key: &str) {
    if let Some(i) = m.iter().position(|(k, _)| *k == key) { m.remove(i); }
}
/// Inverse d'un appariage (ordre d'insertion preserve).
#[allow(dead_code)]
fn inverse(m: &Matching) -> Matching {
    let mut out: Matching = Vec::new();
    for (k, v) in m { match_set(&mut out, v, k); }
    out
}

/// Repr Python d'un appariage (dict) : {'m3': 'w3', ...}
#[allow(dead_code)]
fn py_dict_repr(m: &Matching) -> String {
    let items: Vec<String> = m.iter().map(|(k, v)| format!("'{}': '{}'", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}
/// Repr Python d'une liste de chaines : ['m3', 'm1', 'm2']
#[allow(dead_code)]
fn py_list_repr(v: &[&'static str]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
/// Repr Python d'un booleen.
#[allow(dead_code)]
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
/// Repr Python d'un couple (proposeur, respondent).
#[allow(dead_code)]
fn py_pair_repr(p: Option<Pair>) -> String {
    match p {
        None => "None".to_string(),
        Some((a, b)) => format!("('{}', '{}')", a, b),
    }
}
/// Egalite de dictionnaires : insensible a l'ordre des cles.
#[allow(dead_code)]
fn dict_eq(a: &Matching, b: &Matching) -> bool {
    a.len() == b.len() && a.iter().all(|(k, v)| match_get(b, k) == Some(*v))
}

/// Permutations dans l'ordre de `itertools.permutations` (ordre lexicographique des positions).
#[allow(dead_code)]
fn permutations<T: Clone>(items: &[T]) -> Vec<Vec<T>> {
    let n = items.len();
    let mut out: Vec<Vec<T>> = Vec::new();
    if n == 0 { out.push(Vec::new()); return out; }
    let mut idx: Vec<usize> = (0..n).collect();
    loop {
        out.push(idx.iter().map(|i| items[*i].clone()).collect());
        let mut i = n - 1;
        while i > 0 && idx[i - 1] >= idx[i] { i -= 1; }
        if i == 0 { break; }
        let mut j = n - 1;
        while idx[j] <= idx[i - 1] { j -= 1; }
        idx.swap(i - 1, j);
        idx[i..].reverse();
    }
    out
}

/// Algorithme de Gale-Shapley (recherche differee / deferred acceptance).
///
/// proposers  : les proposeurs, dans l'ordre de traitement (pile LIFO comme `libres.pop()`).
/// prop_prefs : preferences des proposeurs ; resp_prefs : preferences des repondants.
#[allow(dead_code)]
fn gale_shapley(proposers: &[&'static str], prop_prefs: &Prefs, _respondents: &[&'static str], resp_prefs: &Prefs) -> Matching {
    let mut actuel: Matching = Vec::new();     // repondant -> proposeur
    let mut appariage: Matching = Vec::new();  // proposeur  -> repondant
    let mut libres: Vec<&'static str> = proposers.to_vec();
    let mut suivant: Vec<(&'static str, usize)> = proposers.iter().map(|p| (*p, 0)).collect();
    while let Some(p) = libres.pop() {
        let pref = prefs_of(prop_prefs, p).clone();
        let mut i = suivant.iter().find(|(k, _)| *k == p).map(|(_, v)| *v).unwrap_or(0);
        while i < pref.len() {
            let r = pref[i];
            i += 1;
            for e in suivant.iter_mut() { if e.0 == p { e.1 = i; } }
            match match_get(&actuel, r) {
                None => {
                    match_set(&mut actuel, r, p);
                    match_set(&mut appariage, p, r);
                    break;
                }
                Some(q) => {
                    if rank_of(resp_prefs, r, p) < rank_of(resp_prefs, r, q) {
                        match_set(&mut actuel, r, p);
                        match_set(&mut appariage, p, r);
                        match_remove(&mut appariage, q);
                        libres.push(q);
                        break;
                    }
                }
            }
        }
    }
    appariage
}

/// Retourne une paire bloquante (proposeur, respondent), ou None si stable.
#[allow(dead_code)]
fn trouver_paire_bloquante(appariage: &Matching, prop_prefs: &Prefs, resp_prefs: &Prefs) -> Option<Pair> {
    let mut actuel: Matching = Vec::new();
    for (p, r) in appariage { match_set(&mut actuel, r, p); }
    for (p, r) in appariage {
        let pref = prefs_of(prop_prefs, *p).clone();
        for r2 in pref {
            if r2 == *r { break; }                       // p ne prefere personne au-dessus de r
            let a = match_get(&actuel, r2).expect("repondant apparie");
            if rank_of(resp_prefs, r2, *p) < rank_of(resp_prefs, r2, a) {
                return Some((*p, r2));                   // p prefere r2, et r2 prefere p a son actuel
            }
        }
    }
    None
}

let hommes = ["m1", "m2", "m3"];
let femmes = ["w1", "w2", "w3"];
let h_prefs: Prefs = vec![
    ("m1", vec!["w1", "w2", "w3"]),
    ("m2", vec!["w2", "w3", "w1"]),
    ("m3", vec!["w3", "w1", "w2"]),
];
let f_prefs: Prefs = vec![
    ("w1", vec!["m3", "m1", "m2"]),
    ("w2", vec!["m1", "m2", "m3"]),
    ("w3", vec!["m2", "m3", "m1"]),
];

let m_opt = gale_shapley(&hommes, &h_prefs, &femmes, &f_prefs);   // hommes proposent
let f_opt = inverse(&gale_shapley(&femmes, &f_prefs, &hommes, &h_prefs)); // femmes proposent

println!("Equilibre 'hommes proposent' : {}", py_dict_repr(&m_opt));
println!("Equilibre 'femmes proposent' : {}", py_dict_repr(&f_opt));
println!("Identiques ? {}", py_bool(dict_eq(&m_opt, &f_opt)));
println!();
println!("Rang du partenaire de chaque homme (1 = son premier choix) :");
for &h in hommes.iter() {
    let r_h = rank_of(&h_prefs, h, match_get(&m_opt, h).expect("apparie")) + 1;
    let r_f = rank_of(&h_prefs, h, match_get(&f_opt, h).expect("apparie")) + 1;
    println!("  {} : {}e si les hommes proposent, {}e si les femmes proposent", h, r_h, r_f);
}

let stables_a: Vec<Matching> = permutations(&femmes.to_vec()).iter().filter_map(|ordre| {
    let mut appariage: Matching = Vec::new();
    for (k, v) in hommes.iter().zip(ordre.iter()) { match_set(&mut appariage, k, v); }
    if trouver_paire_bloquante(&appariage, &h_prefs, &f_prefs).is_none() { Some(appariage) } else { None }
}).collect();
println!("Oracle indépendant : {} appariages stables sur 6 bijections pour A.", stables_a.len());

Equilibre 'hommes proposent' : {'m3': 'w3', 'm2': 'w2', 'm1': 'w1'}


Equilibre 'femmes proposent' : {'m2': 'w3', 'm1': 'w2', 'm3': 'w1'}


Identiques ? False


Rang du partenaire de chaque homme (1 = son premier choix) :


  m1 : 1e si les hommes proposent, 2e si les femmes proposent


  m2 : 1e si les hommes proposent, 2e si les femmes proposent


  m3 : 1e si les hommes proposent, 2e si les femmes proposent


Oracle indépendant : 2 appariages stables sur 6 bijections pour A.


### Résultat 2 — L'avantage du côté proposeur

La sortie montre **deux appariements stables distincts** sur la même instance : `{m1: w1, m2: w2, m3: w3}` quand les hommes proposent, `{m1: w2, m2: w3, m3: w1}` quand les femmes proposent (`Identiques ? False`).

Le tableau de rangs imprimé est la lecture clé : **chaque** homme passe du 1er choix quand il propose au 2e quand il ne propose plus — aucun n'est indifférent au rôle. En croisant ce tableau avec les listes affichées en section 1, la contrepartie côté femmes est le **miroir exact** : dans l'équilibre « hommes proposent », `w1`, `w2` et `w3` sont toutes à leur **2e choix** ; dans l'équilibre « femmes proposent », elles sont toutes à leur **1er**. L'instance A réalise ainsi le cas d'école complet : le côté proposeur obtient son **meilleur** appariement stable, le côté répondant son **pire** (théorème d'**optimalité** de Gale–Shapley). *Dans la conception du mécanisme, donner le pouvoir de proposer avantage celui qui propose.*

Formulé en général, ce théorème (souvent attribué à Gale–Shapley–Roth) dit plus que cette instance ne montre : quand les hommes proposent, **chaque** homme obtient le **meilleur** partenaire qu'il peut espérer parmi **tous** les appariements stables, et **chaque** femme son **pire**. « Meilleur » et « pire » se comparent sur l'ensemble **complet** des équilibres stables, pas seulement sur les deux que nous voyons ici. Sur l'instance A il n'y en a que deux, si bien que « meilleur » et « pire » tombent exactement sur les deux extrêmes observés. Mais sur des instances plus grandes, le nombre d'équilibres stables peut exploser (il n'existe pas de borne polynomiale sur ce nombre), et l'écart entre le meilleur et le pire stable d'un même agent devient considérable. C'est ce qui donne tout son poids à la règle « qui propose » : l'avantage du proposeur n'est pas un artifact de symétrie de cet exemple, c'est une **structure** du mécanisme.


### Témoin exécuté — l'optimalité du côté proposeur, vérifiée par calcul

La cellule précédente énonce le théorème d'optimalité de Gale–Shapley — *quand les hommes proposent, chaque homme obtient son meilleur partenaire stable, chaque femme son pire* — et l'argumente à partir du comptage : l'instance A n'admet que **deux** appariements stables, donc les deux extrêmes observés **sont** le meilleur et le pire. Ce témoin **calcule** la prédiction au lieu de l'inférer : il parcourt les `stables_A` énumérés par l'oracle et vérifie que `m_opt` donne à chaque agent son rang minimal et `f_opt` son rang maximal, dans les deux directions.

In [7]:
type Pair = (&'static str, &'static str);
type Prefs = Vec<(&'static str, Vec<&'static str>)>;
type Matching = Vec<(&'static str, &'static str)>;

/// Preferences d'un agent, dans l'ordre declare.
#[allow(dead_code)]
fn prefs_of<'a>(prefs: &'a Prefs, agent: &str) -> &'a Vec<&'static str> {
    prefs.iter().find(|(k, _)| *k == agent).map(|(_, v)| v).expect("agent absent des preferences")
}
/// Cles d'un registre de preferences, dans l'ordre d'insertion.
#[allow(dead_code)]
fn keys(prefs: &Prefs) -> Vec<&'static str> { prefs.iter().map(|(k, _)| *k).collect() }
/// Copie des preferences (les listes sont copiees, comme `copie_prefs`).
#[allow(dead_code)]
fn copie_prefs(prefs: &Prefs) -> Prefs { prefs.iter().map(|(k, v)| (*k, v.clone())).collect() }
/// Rang (0 = premier choix) d'un element dans la liste d'un agent.
#[allow(dead_code)]
fn rank_of(prefs: &Prefs, agent: &str, item: &str) -> usize {
    prefs_of(prefs, agent).iter().position(|x| *x == item).expect("element absent de la liste")
}

/// Partenaire d'une cle dans un appariage (None si absent).
#[allow(dead_code)]
fn match_get(m: &Matching, key: &str) -> Option<&'static str> {
    m.iter().find(|(k, _)| *k == key).map(|(_, v)| *v)
}
/// Ecriture dans un appariage : l'ordre d'insertion est preserve (dict Python).
#[allow(dead_code)]
fn match_set(m: &mut Matching, key: &'static str, value: &'static str) {
    match m.iter_mut().find(|(k, _)| *k == key) {
        Some(entry) => entry.1 = value,
        None => m.push((key, value)),
    }
}
/// Suppression d'une cle (comme `del` sur un dict).
#[allow(dead_code)]
fn match_remove(m: &mut Matching, key: &str) {
    if let Some(i) = m.iter().position(|(k, _)| *k == key) { m.remove(i); }
}
/// Inverse d'un appariage (ordre d'insertion preserve).
#[allow(dead_code)]
fn inverse(m: &Matching) -> Matching {
    let mut out: Matching = Vec::new();
    for (k, v) in m { match_set(&mut out, v, k); }
    out
}

/// Repr Python d'un appariage (dict) : {'m3': 'w3', ...}
#[allow(dead_code)]
fn py_dict_repr(m: &Matching) -> String {
    let items: Vec<String> = m.iter().map(|(k, v)| format!("'{}': '{}'", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}
/// Repr Python d'une liste de chaines : ['m3', 'm1', 'm2']
#[allow(dead_code)]
fn py_list_repr(v: &[&'static str]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
/// Repr Python d'un booleen.
#[allow(dead_code)]
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
/// Repr Python d'un couple (proposeur, respondent).
#[allow(dead_code)]
fn py_pair_repr(p: Option<Pair>) -> String {
    match p {
        None => "None".to_string(),
        Some((a, b)) => format!("('{}', '{}')", a, b),
    }
}
/// Egalite de dictionnaires : insensible a l'ordre des cles.
#[allow(dead_code)]
fn dict_eq(a: &Matching, b: &Matching) -> bool {
    a.len() == b.len() && a.iter().all(|(k, v)| match_get(b, k) == Some(*v))
}

/// Permutations dans l'ordre de `itertools.permutations` (ordre lexicographique des positions).
#[allow(dead_code)]
fn permutations<T: Clone>(items: &[T]) -> Vec<Vec<T>> {
    let n = items.len();
    let mut out: Vec<Vec<T>> = Vec::new();
    if n == 0 { out.push(Vec::new()); return out; }
    let mut idx: Vec<usize> = (0..n).collect();
    loop {
        out.push(idx.iter().map(|i| items[*i].clone()).collect());
        let mut i = n - 1;
        while i > 0 && idx[i - 1] >= idx[i] { i -= 1; }
        if i == 0 { break; }
        let mut j = n - 1;
        while idx[j] <= idx[i - 1] { j -= 1; }
        idx.swap(i - 1, j);
        idx[i..].reverse();
    }
    out
}

/// Algorithme de Gale-Shapley (recherche differee / deferred acceptance).
///
/// proposers  : les proposeurs, dans l'ordre de traitement (pile LIFO comme `libres.pop()`).
/// prop_prefs : preferences des proposeurs ; resp_prefs : preferences des repondants.
#[allow(dead_code)]
fn gale_shapley(proposers: &[&'static str], prop_prefs: &Prefs, _respondents: &[&'static str], resp_prefs: &Prefs) -> Matching {
    let mut actuel: Matching = Vec::new();     // repondant -> proposeur
    let mut appariage: Matching = Vec::new();  // proposeur  -> repondant
    let mut libres: Vec<&'static str> = proposers.to_vec();
    let mut suivant: Vec<(&'static str, usize)> = proposers.iter().map(|p| (*p, 0)).collect();
    while let Some(p) = libres.pop() {
        let pref = prefs_of(prop_prefs, p).clone();
        let mut i = suivant.iter().find(|(k, _)| *k == p).map(|(_, v)| *v).unwrap_or(0);
        while i < pref.len() {
            let r = pref[i];
            i += 1;
            for e in suivant.iter_mut() { if e.0 == p { e.1 = i; } }
            match match_get(&actuel, r) {
                None => {
                    match_set(&mut actuel, r, p);
                    match_set(&mut appariage, p, r);
                    break;
                }
                Some(q) => {
                    if rank_of(resp_prefs, r, p) < rank_of(resp_prefs, r, q) {
                        match_set(&mut actuel, r, p);
                        match_set(&mut appariage, p, r);
                        match_remove(&mut appariage, q);
                        libres.push(q);
                        break;
                    }
                }
            }
        }
    }
    appariage
}

/// Retourne une paire bloquante (proposeur, respondent), ou None si stable.
#[allow(dead_code)]
fn trouver_paire_bloquante(appariage: &Matching, prop_prefs: &Prefs, resp_prefs: &Prefs) -> Option<Pair> {
    let mut actuel: Matching = Vec::new();
    for (p, r) in appariage { match_set(&mut actuel, r, p); }
    for (p, r) in appariage {
        let pref = prefs_of(prop_prefs, *p).clone();
        for r2 in pref {
            if r2 == *r { break; }                       // p ne prefere personne au-dessus de r
            let a = match_get(&actuel, r2).expect("repondant apparie");
            if rank_of(resp_prefs, r2, *p) < rank_of(resp_prefs, r2, a) {
                return Some((*p, r2));                   // p prefere r2, et r2 prefere p a son actuel
            }
        }
    }
    None
}

let hommes = ["m1", "m2", "m3"];
let femmes = ["w1", "w2", "w3"];
let h_prefs: Prefs = vec![
    ("m1", vec!["w1", "w2", "w3"]),
    ("m2", vec!["w2", "w3", "w1"]),
    ("m3", vec!["w3", "w1", "w2"]),
];
let f_prefs: Prefs = vec![
    ("w1", vec!["m3", "m1", "m2"]),
    ("w2", vec!["m1", "m2", "m3"]),
    ("w3", vec!["m2", "m3", "m1"]),
];

/// Partenaire d'un agent dans un appariage {homme: femme}.
#[allow(dead_code)]
fn part_de(s: &Matching, agent: &str) -> &'static str {
    match match_get(s, agent) {
        Some(v) => v,
        None => s.iter().find(|(_, f)| *f == agent).map(|(h, _)| *h).expect("agent absent de l appariage"),
    }
}

let stables_a: Vec<Matching> = permutations(&femmes.to_vec()).iter().filter_map(|ordre| {
    let mut appariage: Matching = Vec::new();
    for (k, v) in hommes.iter().zip(ordre.iter()) { match_set(&mut appariage, k, v); }
    if trouver_paire_bloquante(&appariage, &h_prefs, &f_prefs).is_none() { Some(appariage) } else { None }
}).collect();
let m_opt = gale_shapley(&hommes, &h_prefs, &femmes, &f_prefs);
let f_opt = inverse(&gale_shapley(&femmes, &f_prefs, &hommes, &h_prefs));

println!("Optimalite des HOMMES sur les {} appariements stables de A :", stables_a.len());
for &h in hommes.iter() {
    let rangaux: Vec<usize> = stables_a.iter().map(|s| rank_of(&h_prefs, h, part_de(s, h)) + 1).collect();
    let best = *rangaux.iter().min().expect("liste non vide");
    let worst = *rangaux.iter().max().expect("liste non vide");
    let ro = rank_of(&h_prefs, h, part_de(&m_opt, h)) + 1;
    let rf = rank_of(&h_prefs, h, part_de(&f_opt, h)) + 1;
    assert!(ro == best, "{}: m_opt ({}) != meilleur stable ({})", h, ro, best);
    assert!(rf == worst, "{}: f_opt ({}) != pire stable ({})", h, rf, worst);
    println!("  {} : m_opt {}e = MIN, f_opt {}e = MAX  (stables : {:?})", h, ro, rf, rangaux);
}

println!("Optimalite des FEMMES (miroir) :");
for &f in femmes.iter() {
    let rangaux: Vec<usize> = stables_a.iter().map(|s| rank_of(&f_prefs, f, part_de(s, f)) + 1).collect();
    let best = *rangaux.iter().min().expect("liste non vide");
    let worst = *rangaux.iter().max().expect("liste non vide");
    let ro = rank_of(&f_prefs, f, part_de(&m_opt, f)) + 1;
    let rf = rank_of(&f_prefs, f, part_de(&f_opt, f)) + 1;
    assert!(ro == worst, "{}: m_opt ({}) != pire stable ({})", f, ro, worst);
    assert!(rf == best, "{}: f_opt ({}) != meilleur stable ({})", f, rf, best);
    println!("  {} : f_opt {}e = MIN, m_opt {}e = MAX  (stables : {:?})", f, rf, ro, rangaux);
};

Optimalite des HOMMES sur les 2 appariements stables de A :


  m1 : m_opt 1e = MIN, f_opt 2e = MAX  (stables : [1, 2])


  m2 : m_opt 1e = MIN, f_opt 2e = MAX  (stables : [1, 2])


  m3 : m_opt 1e = MIN, f_opt 2e = MAX  (stables : [1, 2])


Optimalite des FEMMES (miroir) :


  w1 : f_opt 1e = MIN, m_opt 2e = MAX  (stables : [2, 1])


  w2 : f_opt 1e = MIN, m_opt 2e = MAX  (stables : [2, 1])


  w3 : f_opt 1e = MIN, m_opt 2e = MAX  (stables : [2, 1])


### Lecture du résultat

Le calcul **confirme et précise** l'argument de la cellule précédente. Le théorème d'optimalité ne reposait ici que sur le comptage (deux stables ⇒ les deux observés sont les extrêmes) ; le témoin l'exhibe désormais par calcul : sur les `stables_A` énumérés, `m_opt` donne à **chaque** homme son rang le plus bas (1er) et `f_opt` son rang le plus haut (2e), et le miroir tient côté femmes. « Meilleur » et « pire » ne sont plus une inférence sur le compte : ce sont des égalités vérifiées sur l'ensemble complet des équilibres stables — c'est précisément la portée générale du théorème, qui ne dépend pas de ce qu'il n'y en a que deux ici.

In [8]:
type Pair = (&'static str, &'static str);
type Prefs = Vec<(&'static str, Vec<&'static str>)>;
type Matching = Vec<(&'static str, &'static str)>;

/// Preferences d'un agent, dans l'ordre declare.
#[allow(dead_code)]
fn prefs_of<'a>(prefs: &'a Prefs, agent: &str) -> &'a Vec<&'static str> {
    prefs.iter().find(|(k, _)| *k == agent).map(|(_, v)| v).expect("agent absent des preferences")
}
/// Cles d'un registre de preferences, dans l'ordre d'insertion.
#[allow(dead_code)]
fn keys(prefs: &Prefs) -> Vec<&'static str> { prefs.iter().map(|(k, _)| *k).collect() }
/// Copie des preferences (les listes sont copiees, comme `copie_prefs`).
#[allow(dead_code)]
fn copie_prefs(prefs: &Prefs) -> Prefs { prefs.iter().map(|(k, v)| (*k, v.clone())).collect() }
/// Rang (0 = premier choix) d'un element dans la liste d'un agent.
#[allow(dead_code)]
fn rank_of(prefs: &Prefs, agent: &str, item: &str) -> usize {
    prefs_of(prefs, agent).iter().position(|x| *x == item).expect("element absent de la liste")
}

/// Partenaire d'une cle dans un appariage (None si absent).
#[allow(dead_code)]
fn match_get(m: &Matching, key: &str) -> Option<&'static str> {
    m.iter().find(|(k, _)| *k == key).map(|(_, v)| *v)
}
/// Ecriture dans un appariage : l'ordre d'insertion est preserve (dict Python).
#[allow(dead_code)]
fn match_set(m: &mut Matching, key: &'static str, value: &'static str) {
    match m.iter_mut().find(|(k, _)| *k == key) {
        Some(entry) => entry.1 = value,
        None => m.push((key, value)),
    }
}
/// Suppression d'une cle (comme `del` sur un dict).
#[allow(dead_code)]
fn match_remove(m: &mut Matching, key: &str) {
    if let Some(i) = m.iter().position(|(k, _)| *k == key) { m.remove(i); }
}
/// Inverse d'un appariage (ordre d'insertion preserve).
#[allow(dead_code)]
fn inverse(m: &Matching) -> Matching {
    let mut out: Matching = Vec::new();
    for (k, v) in m { match_set(&mut out, v, k); }
    out
}

/// Repr Python d'un appariage (dict) : {'m3': 'w3', ...}
#[allow(dead_code)]
fn py_dict_repr(m: &Matching) -> String {
    let items: Vec<String> = m.iter().map(|(k, v)| format!("'{}': '{}'", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}
/// Repr Python d'une liste de chaines : ['m3', 'm1', 'm2']
#[allow(dead_code)]
fn py_list_repr(v: &[&'static str]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
/// Repr Python d'un booleen.
#[allow(dead_code)]
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
/// Repr Python d'un couple (proposeur, respondent).
#[allow(dead_code)]
fn py_pair_repr(p: Option<Pair>) -> String {
    match p {
        None => "None".to_string(),
        Some((a, b)) => format!("('{}', '{}')", a, b),
    }
}
/// Egalite de dictionnaires : insensible a l'ordre des cles.
#[allow(dead_code)]
fn dict_eq(a: &Matching, b: &Matching) -> bool {
    a.len() == b.len() && a.iter().all(|(k, v)| match_get(b, k) == Some(*v))
}

/// Permutations dans l'ordre de `itertools.permutations` (ordre lexicographique des positions).
#[allow(dead_code)]
fn permutations<T: Clone>(items: &[T]) -> Vec<Vec<T>> {
    let n = items.len();
    let mut out: Vec<Vec<T>> = Vec::new();
    if n == 0 { out.push(Vec::new()); return out; }
    let mut idx: Vec<usize> = (0..n).collect();
    loop {
        out.push(idx.iter().map(|i| items[*i].clone()).collect());
        let mut i = n - 1;
        while i > 0 && idx[i - 1] >= idx[i] { i -= 1; }
        if i == 0 { break; }
        let mut j = n - 1;
        while idx[j] <= idx[i - 1] { j -= 1; }
        idx.swap(i - 1, j);
        idx[i..].reverse();
    }
    out
}

/// Algorithme de Gale-Shapley (recherche differee / deferred acceptance).
///
/// proposers  : les proposeurs, dans l'ordre de traitement (pile LIFO comme `libres.pop()`).
/// prop_prefs : preferences des proposeurs ; resp_prefs : preferences des repondants.
#[allow(dead_code)]
fn gale_shapley(proposers: &[&'static str], prop_prefs: &Prefs, _respondents: &[&'static str], resp_prefs: &Prefs) -> Matching {
    let mut actuel: Matching = Vec::new();     // repondant -> proposeur
    let mut appariage: Matching = Vec::new();  // proposeur  -> repondant
    let mut libres: Vec<&'static str> = proposers.to_vec();
    let mut suivant: Vec<(&'static str, usize)> = proposers.iter().map(|p| (*p, 0)).collect();
    while let Some(p) = libres.pop() {
        let pref = prefs_of(prop_prefs, p).clone();
        let mut i = suivant.iter().find(|(k, _)| *k == p).map(|(_, v)| *v).unwrap_or(0);
        while i < pref.len() {
            let r = pref[i];
            i += 1;
            for e in suivant.iter_mut() { if e.0 == p { e.1 = i; } }
            match match_get(&actuel, r) {
                None => {
                    match_set(&mut actuel, r, p);
                    match_set(&mut appariage, p, r);
                    break;
                }
                Some(q) => {
                    if rank_of(resp_prefs, r, p) < rank_of(resp_prefs, r, q) {
                        match_set(&mut actuel, r, p);
                        match_set(&mut appariage, p, r);
                        match_remove(&mut appariage, q);
                        libres.push(q);
                        break;
                    }
                }
            }
        }
    }
    appariage
}

/// Retourne une paire bloquante (proposeur, respondent), ou None si stable.
#[allow(dead_code)]
fn trouver_paire_bloquante(appariage: &Matching, prop_prefs: &Prefs, resp_prefs: &Prefs) -> Option<Pair> {
    let mut actuel: Matching = Vec::new();
    for (p, r) in appariage { match_set(&mut actuel, r, p); }
    for (p, r) in appariage {
        let pref = prefs_of(prop_prefs, *p).clone();
        for r2 in pref {
            if r2 == *r { break; }                       // p ne prefere personne au-dessus de r
            let a = match_get(&actuel, r2).expect("repondant apparie");
            if rank_of(resp_prefs, r2, *p) < rank_of(resp_prefs, r2, a) {
                return Some((*p, r2));                   // p prefere r2, et r2 prefere p a son actuel
            }
        }
    }
    None
}

let c_h: Prefs = vec![
    ("m1", vec!["w2", "w1", "w3"]),
    ("m2", vec!["w2", "w3", "w1"]),
    ("m3", vec!["w1", "w2", "w3"]),
];
let c_f: Prefs = vec![
    ("w1", vec!["m2", "m1", "m3"]),
    ("w2", vec!["m3", "m1", "m2"]),
    ("w3", vec!["m1", "m2", "m3"]),
];

let c_honnete = gale_shapley(&keys(&c_h), &c_h, &keys(&c_f), &c_f);
println!("Appariage honnete (hommes proposent) : {}", py_dict_repr(&c_honnete));

// (A) Un HOMME peut-il s'ameliorer en mentant ? Recherche EXHAUSTIVE.
let mut hommes = 0usize;
for &h in keys(&c_h).iter() {
    let base = rank_of(&c_h, h, match_get(&c_honnete, h).expect("apparie")) + 1;
    for perm in permutations(&prefs_of(&c_h, h).clone()) {
        let mut p2 = copie_prefs(&c_h);
        for e in p2.iter_mut() { if e.0 == h { e.1 = perm.clone(); } }
        let m2 = gale_shapley(&keys(&c_h), &p2, &keys(&c_f), &c_f);
        if rank_of(&c_h, h, match_get(&m2, h).expect("apparie")) + 1 < base {
            hommes += 1;
            break;
        }
    }
}
println!();
println!("(A) Hommes pouvant s'ameliorer en mentant : {} sur {}", hommes, c_h.len());

Appariage honnete (hommes proposent) : {'m3': 'w1', 'm1': 'w2', 'm2': 'w3'}


(A) Hommes pouvant s'ameliorer en mentant : 0 sur 3


### (B) Examiner les déclarations des répondantes

La cellule précédente énumère les six permutations déclarées pour chacun des trois hommes. On effectue maintenant la même recherche, répondante par répondante, en évaluant les partenaires selon les préférences vraies.

In [9]:
type Pair = (&'static str, &'static str);
type Prefs = Vec<(&'static str, Vec<&'static str>)>;
type Matching = Vec<(&'static str, &'static str)>;

/// Preferences d'un agent, dans l'ordre declare.
#[allow(dead_code)]
fn prefs_of<'a>(prefs: &'a Prefs, agent: &str) -> &'a Vec<&'static str> {
    prefs.iter().find(|(k, _)| *k == agent).map(|(_, v)| v).expect("agent absent des preferences")
}
/// Cles d'un registre de preferences, dans l'ordre d'insertion.
#[allow(dead_code)]
fn keys(prefs: &Prefs) -> Vec<&'static str> { prefs.iter().map(|(k, _)| *k).collect() }
/// Copie des preferences (les listes sont copiees, comme `copie_prefs`).
#[allow(dead_code)]
fn copie_prefs(prefs: &Prefs) -> Prefs { prefs.iter().map(|(k, v)| (*k, v.clone())).collect() }
/// Rang (0 = premier choix) d'un element dans la liste d'un agent.
#[allow(dead_code)]
fn rank_of(prefs: &Prefs, agent: &str, item: &str) -> usize {
    prefs_of(prefs, agent).iter().position(|x| *x == item).expect("element absent de la liste")
}

/// Partenaire d'une cle dans un appariage (None si absent).
#[allow(dead_code)]
fn match_get(m: &Matching, key: &str) -> Option<&'static str> {
    m.iter().find(|(k, _)| *k == key).map(|(_, v)| *v)
}
/// Ecriture dans un appariage : l'ordre d'insertion est preserve (dict Python).
#[allow(dead_code)]
fn match_set(m: &mut Matching, key: &'static str, value: &'static str) {
    match m.iter_mut().find(|(k, _)| *k == key) {
        Some(entry) => entry.1 = value,
        None => m.push((key, value)),
    }
}
/// Suppression d'une cle (comme `del` sur un dict).
#[allow(dead_code)]
fn match_remove(m: &mut Matching, key: &str) {
    if let Some(i) = m.iter().position(|(k, _)| *k == key) { m.remove(i); }
}
/// Inverse d'un appariage (ordre d'insertion preserve).
#[allow(dead_code)]
fn inverse(m: &Matching) -> Matching {
    let mut out: Matching = Vec::new();
    for (k, v) in m { match_set(&mut out, v, k); }
    out
}

/// Repr Python d'un appariage (dict) : {'m3': 'w3', ...}
#[allow(dead_code)]
fn py_dict_repr(m: &Matching) -> String {
    let items: Vec<String> = m.iter().map(|(k, v)| format!("'{}': '{}'", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}
/// Repr Python d'une liste de chaines : ['m3', 'm1', 'm2']
#[allow(dead_code)]
fn py_list_repr(v: &[&'static str]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
/// Repr Python d'un booleen.
#[allow(dead_code)]
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
/// Repr Python d'un couple (proposeur, respondent).
#[allow(dead_code)]
fn py_pair_repr(p: Option<Pair>) -> String {
    match p {
        None => "None".to_string(),
        Some((a, b)) => format!("('{}', '{}')", a, b),
    }
}
/// Egalite de dictionnaires : insensible a l'ordre des cles.
#[allow(dead_code)]
fn dict_eq(a: &Matching, b: &Matching) -> bool {
    a.len() == b.len() && a.iter().all(|(k, v)| match_get(b, k) == Some(*v))
}

/// Permutations dans l'ordre de `itertools.permutations` (ordre lexicographique des positions).
#[allow(dead_code)]
fn permutations<T: Clone>(items: &[T]) -> Vec<Vec<T>> {
    let n = items.len();
    let mut out: Vec<Vec<T>> = Vec::new();
    if n == 0 { out.push(Vec::new()); return out; }
    let mut idx: Vec<usize> = (0..n).collect();
    loop {
        out.push(idx.iter().map(|i| items[*i].clone()).collect());
        let mut i = n - 1;
        while i > 0 && idx[i - 1] >= idx[i] { i -= 1; }
        if i == 0 { break; }
        let mut j = n - 1;
        while idx[j] <= idx[i - 1] { j -= 1; }
        idx.swap(i - 1, j);
        idx[i..].reverse();
    }
    out
}

/// Algorithme de Gale-Shapley (recherche differee / deferred acceptance).
///
/// proposers  : les proposeurs, dans l'ordre de traitement (pile LIFO comme `libres.pop()`).
/// prop_prefs : preferences des proposeurs ; resp_prefs : preferences des repondants.
#[allow(dead_code)]
fn gale_shapley(proposers: &[&'static str], prop_prefs: &Prefs, _respondents: &[&'static str], resp_prefs: &Prefs) -> Matching {
    let mut actuel: Matching = Vec::new();     // repondant -> proposeur
    let mut appariage: Matching = Vec::new();  // proposeur  -> repondant
    let mut libres: Vec<&'static str> = proposers.to_vec();
    let mut suivant: Vec<(&'static str, usize)> = proposers.iter().map(|p| (*p, 0)).collect();
    while let Some(p) = libres.pop() {
        let pref = prefs_of(prop_prefs, p).clone();
        let mut i = suivant.iter().find(|(k, _)| *k == p).map(|(_, v)| *v).unwrap_or(0);
        while i < pref.len() {
            let r = pref[i];
            i += 1;
            for e in suivant.iter_mut() { if e.0 == p { e.1 = i; } }
            match match_get(&actuel, r) {
                None => {
                    match_set(&mut actuel, r, p);
                    match_set(&mut appariage, p, r);
                    break;
                }
                Some(q) => {
                    if rank_of(resp_prefs, r, p) < rank_of(resp_prefs, r, q) {
                        match_set(&mut actuel, r, p);
                        match_set(&mut appariage, p, r);
                        match_remove(&mut appariage, q);
                        libres.push(q);
                        break;
                    }
                }
            }
        }
    }
    appariage
}

/// Retourne une paire bloquante (proposeur, respondent), ou None si stable.
#[allow(dead_code)]
fn trouver_paire_bloquante(appariage: &Matching, prop_prefs: &Prefs, resp_prefs: &Prefs) -> Option<Pair> {
    let mut actuel: Matching = Vec::new();
    for (p, r) in appariage { match_set(&mut actuel, r, p); }
    for (p, r) in appariage {
        let pref = prefs_of(prop_prefs, *p).clone();
        for r2 in pref {
            if r2 == *r { break; }                       // p ne prefere personne au-dessus de r
            let a = match_get(&actuel, r2).expect("repondant apparie");
            if rank_of(resp_prefs, r2, *p) < rank_of(resp_prefs, r2, a) {
                return Some((*p, r2));                   // p prefere r2, et r2 prefere p a son actuel
            }
        }
    }
    None
}

let c_h: Prefs = vec![
    ("m1", vec!["w2", "w1", "w3"]),
    ("m2", vec!["w2", "w3", "w1"]),
    ("m3", vec!["w1", "w2", "w3"]),
];
let c_f: Prefs = vec![
    ("w1", vec!["m2", "m1", "m3"]),
    ("w2", vec!["m3", "m1", "m2"]),
    ("w3", vec!["m1", "m2", "m3"]),
];

let c_honnete = gale_shapley(&keys(&c_h), &c_h, &keys(&c_f), &c_f);
let part = inverse(&c_honnete);
let mut trouve: Option<(&'static str, Vec<&'static str>, &'static str, usize, &'static str, usize)> = None;
for &f in keys(&c_f).iter() {
    let base = rank_of(&c_f, f, match_get(&part, f).expect("apparie")) + 1;
    for perm in permutations(&prefs_of(&c_f, f).clone()) {
        let mut fp2 = copie_prefs(&c_f);
        for e in fp2.iter_mut() { if e.0 == f { e.1 = perm.clone(); } }
        let m2 = gale_shapley(&keys(&c_h), &c_h, &keys(&c_f), &fp2);
        let p2 = inverse(&m2);
        let nouveau = rank_of(&c_f, f, match_get(&p2, f).expect("apparie")) + 1;
        if nouveau < base {
            trouve = Some((f, perm.clone(), match_get(&part, f).expect("apparie"), base,
                           match_get(&p2, f).expect("apparie"), nouveau));
            break;
        }
    }
    if trouve.is_some() { break; }
}

match &trouve {
    Some((f, decl, h0, r0, h1, r1)) => {
        println!("(B) {} (liste vraie : {})", f, py_list_repr(&prefs_of(&c_f, f).clone()));
        println!("    honnête : {} (son rang {})  ->  declare {}  ->  {} (son rang {}) : MEILLEUR",
                 h0, r0, py_list_repr(decl), h1, r1);
    }
    None => println!("(B) aucune femme ne peut s'ameliorer ici."),
}

let stables_c: Vec<Matching> = permutations(&keys(&c_f)).iter().filter_map(|ordre| {
    let mut appariage: Matching = Vec::new();
    for (k, v) in keys(&c_h).iter().zip(ordre.iter()) { match_set(&mut appariage, k, v); }
    if trouver_paire_bloquante(&appariage, &c_h, &c_f).is_none() { Some(appariage) } else { None }
}).collect();
println!("Oracle indépendant : {} appariages stables sur 6 pour C.", stables_c.len());

if let Some((f, decl, _, _, _, _)) = trouve {
    let mut prefs_declarees = copie_prefs(&c_f);
    for e in prefs_declarees.iter_mut() { if e.0 == f { e.1 = decl.clone(); } }
    let hommes_modifies = gale_shapley(&keys(&c_h), &c_h, &keys(&c_f), &prefs_declarees);
    let femmes_modifiees = inverse(&hommes_modifies);
    let rangs_f: Vec<String> = keys(&c_f).iter().map(|w| format!(
        "'{}': ({}, {})", w,
        rank_of(&c_f, w, match_get(&part, w).expect("apparie")) + 1,
        rank_of(&c_f, w, match_get(&femmes_modifiees, w).expect("apparie")) + 1)).collect();
    println!("Rangs vrais femmes : {{{}}}", rangs_f.join(", "));
    let rangs_h: Vec<String> = keys(&c_h).iter().map(|h| format!(
        "'{}': ({}, {})", h,
        rank_of(&c_h, h, match_get(&c_honnete, h).expect("apparie")) + 1,
        rank_of(&c_h, h, match_get(&hommes_modifies, h).expect("apparie")) + 1)).collect();
    println!("Rangs vrais hommes : {{{}}}", rangs_h.join(", "));
};

(B) w2 (liste vraie : ['m3', 'm1', 'm2'])


    honnête : m1 (son rang 2)  ->  declare ['m3', 'm2', 'm1']  ->  m3 (son rang 1) : MEILLEUR


Oracle indépendant : 3 appariages stables sur 6 pour C.


Rangs vrais femmes : {'w1': (3, 2), 'w2': (2, 1), 'w3': (2, 2)}


Rangs vrais hommes : {'m1': (1, 2), 'm2': (2, 2), 'm3': (1, 2)}


### Résultat 3 — La non-stratégicité : qui peut tricher ?

La mesure **(A)** est surprenante : **aucun homme** ne peut améliorer son appariement en mentant sur ses préférences (`0 sur 3`, recherche exhaustive sur toutes les permutations de chaque liste). Le théorème d’optimalité du proposant porte sur les appariages stables du modèle classique. La recherche exhaustive ici ne porte que sur six déclarations par homme et cette instance 3×3. Le **côté proposeur est protégé**.

La mesure **(B)** révèle l'asymétrie : `w2`, honnête, obtient `m1` (son rang 2) ; en déclarant `[m3, m2, m1]` — elle ne fait que **repousser `m1` tout en bas** de sa liste — elle obtient `m3`, son rang 1. Le détail vaut une lecture : le partenaire gagné (`m3`) est le meilleur partenaire stable vrai de `w2` dans cette instance finie. Ce constat d’instance ne prouve pas une borne générale sur les manipulations des répondantes. Cette expérience finie montre une possibilité de gain pour une répondante dans C ; elle ne démontre pas une règle générale de manipulation pour le côté répondant.

La protection unilatérale du proposant est un résultat stratégique du modèle classique : listes strictes complètes, appariage un à un, tous les autres agents déclarant leurs préférences vraies. Elle est distincte de l'optimalité parmi les appariages stables. Après un mensonge, la stabilité garantie porte sur les **listes déclarées** ; elle ne suffit donc pas à déduire cette protection depuis l'optimalité au profil vrai. Notre énumération vérifie seulement les déclarations de C. Côté répondant, la liste de w2 modifie les rejets et les propositions ultérieures ; le gain calculé fournit ici un contre-exemple à la protection des répondantes.

L'algorithme n'est donc pas *sans stratagème* (*not strategy-proof*), mais **uniquement du côté des répondants**. D'où le principe de conception : **donner le pouvoir de proposer au côté qu'on veut protéger**. Le NRMP utilise depuis 1998 une variante résidents-proposants ; quotas et couples du dispositif réel dépassent le modèle biparti simple étudié ici.



### Exercice 3 — Rechercher un mensonge profitable
<a id="exercice-3"></a>

Pour une répondante, énumérez ses déclarations complètes et comparez le partenaire obtenu à sa préférence vraie. Une absence de gain doit rester distincte d’un stub non rempli.

In [10]:
type Pair = (&'static str, &'static str);
type Prefs = Vec<(&'static str, Vec<&'static str>)>;
type Matching = Vec<(&'static str, &'static str)>;

/// Preferences d'un agent, dans l'ordre declare.
#[allow(dead_code)]
fn prefs_of<'a>(prefs: &'a Prefs, agent: &str) -> &'a Vec<&'static str> {
    prefs.iter().find(|(k, _)| *k == agent).map(|(_, v)| v).expect("agent absent des preferences")
}
/// Cles d'un registre de preferences, dans l'ordre d'insertion.
#[allow(dead_code)]
fn keys(prefs: &Prefs) -> Vec<&'static str> { prefs.iter().map(|(k, _)| *k).collect() }
/// Copie des preferences (les listes sont copiees, comme `copie_prefs`).
#[allow(dead_code)]
fn copie_prefs(prefs: &Prefs) -> Prefs { prefs.iter().map(|(k, v)| (*k, v.clone())).collect() }
/// Rang (0 = premier choix) d'un element dans la liste d'un agent.
#[allow(dead_code)]
fn rank_of(prefs: &Prefs, agent: &str, item: &str) -> usize {
    prefs_of(prefs, agent).iter().position(|x| *x == item).expect("element absent de la liste")
}

/// Partenaire d'une cle dans un appariage (None si absent).
#[allow(dead_code)]
fn match_get(m: &Matching, key: &str) -> Option<&'static str> {
    m.iter().find(|(k, _)| *k == key).map(|(_, v)| *v)
}
/// Ecriture dans un appariage : l'ordre d'insertion est preserve (dict Python).
#[allow(dead_code)]
fn match_set(m: &mut Matching, key: &'static str, value: &'static str) {
    match m.iter_mut().find(|(k, _)| *k == key) {
        Some(entry) => entry.1 = value,
        None => m.push((key, value)),
    }
}
/// Suppression d'une cle (comme `del` sur un dict).
#[allow(dead_code)]
fn match_remove(m: &mut Matching, key: &str) {
    if let Some(i) = m.iter().position(|(k, _)| *k == key) { m.remove(i); }
}
/// Inverse d'un appariage (ordre d'insertion preserve).
#[allow(dead_code)]
fn inverse(m: &Matching) -> Matching {
    let mut out: Matching = Vec::new();
    for (k, v) in m { match_set(&mut out, v, k); }
    out
}

/// Repr Python d'un appariage (dict) : {'m3': 'w3', ...}
#[allow(dead_code)]
fn py_dict_repr(m: &Matching) -> String {
    let items: Vec<String> = m.iter().map(|(k, v)| format!("'{}': '{}'", k, v)).collect();
    format!("{{{}}}", items.join(", "))
}
/// Repr Python d'une liste de chaines : ['m3', 'm1', 'm2']
#[allow(dead_code)]
fn py_list_repr(v: &[&'static str]) -> String {
    let items: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", items.join(", "))
}
/// Repr Python d'un booleen.
#[allow(dead_code)]
fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }
/// Repr Python d'un couple (proposeur, respondent).
#[allow(dead_code)]
fn py_pair_repr(p: Option<Pair>) -> String {
    match p {
        None => "None".to_string(),
        Some((a, b)) => format!("('{}', '{}')", a, b),
    }
}
/// Egalite de dictionnaires : insensible a l'ordre des cles.
#[allow(dead_code)]
fn dict_eq(a: &Matching, b: &Matching) -> bool {
    a.len() == b.len() && a.iter().all(|(k, v)| match_get(b, k) == Some(*v))
}

/// Permutations dans l'ordre de `itertools.permutations` (ordre lexicographique des positions).
#[allow(dead_code)]
fn permutations<T: Clone>(items: &[T]) -> Vec<Vec<T>> {
    let n = items.len();
    let mut out: Vec<Vec<T>> = Vec::new();
    if n == 0 { out.push(Vec::new()); return out; }
    let mut idx: Vec<usize> = (0..n).collect();
    loop {
        out.push(idx.iter().map(|i| items[*i].clone()).collect());
        let mut i = n - 1;
        while i > 0 && idx[i - 1] >= idx[i] { i -= 1; }
        if i == 0 { break; }
        let mut j = n - 1;
        while idx[j] <= idx[i - 1] { j -= 1; }
        idx.swap(i - 1, j);
        idx[i..].reverse();
    }
    out
}

/// Algorithme de Gale-Shapley (recherche differee / deferred acceptance).
///
/// proposers  : les proposeurs, dans l'ordre de traitement (pile LIFO comme `libres.pop()`).
/// prop_prefs : preferences des proposeurs ; resp_prefs : preferences des repondants.
#[allow(dead_code)]
fn gale_shapley(proposers: &[&'static str], prop_prefs: &Prefs, _respondents: &[&'static str], resp_prefs: &Prefs) -> Matching {
    let mut actuel: Matching = Vec::new();     // repondant -> proposeur
    let mut appariage: Matching = Vec::new();  // proposeur  -> repondant
    let mut libres: Vec<&'static str> = proposers.to_vec();
    let mut suivant: Vec<(&'static str, usize)> = proposers.iter().map(|p| (*p, 0)).collect();
    while let Some(p) = libres.pop() {
        let pref = prefs_of(prop_prefs, p).clone();
        let mut i = suivant.iter().find(|(k, _)| *k == p).map(|(_, v)| *v).unwrap_or(0);
        while i < pref.len() {
            let r = pref[i];
            i += 1;
            for e in suivant.iter_mut() { if e.0 == p { e.1 = i; } }
            match match_get(&actuel, r) {
                None => {
                    match_set(&mut actuel, r, p);
                    match_set(&mut appariage, p, r);
                    break;
                }
                Some(q) => {
                    if rank_of(resp_prefs, r, p) < rank_of(resp_prefs, r, q) {
                        match_set(&mut actuel, r, p);
                        match_set(&mut appariage, p, r);
                        match_remove(&mut appariage, q);
                        libres.push(q);
                        break;
                    }
                }
            }
        }
    }
    appariage
}

/// Retourne une paire bloquante (proposeur, respondent), ou None si stable.
#[allow(dead_code)]
fn trouver_paire_bloquante(appariage: &Matching, prop_prefs: &Prefs, resp_prefs: &Prefs) -> Option<Pair> {
    let mut actuel: Matching = Vec::new();
    for (p, r) in appariage { match_set(&mut actuel, r, p); }
    for (p, r) in appariage {
        let pref = prefs_of(prop_prefs, *p).clone();
        for r2 in pref {
            if r2 == *r { break; }                       // p ne prefere personne au-dessus de r
            let a = match_get(&actuel, r2).expect("repondant apparie");
            if rank_of(resp_prefs, r2, *p) < rank_of(resp_prefs, r2, a) {
                return Some((*p, r2));                   // p prefere r2, et r2 prefere p a son actuel
            }
        }
    }
    None
}

let hommes = ["m1", "m2", "m3"];
let femmes = ["w1", "w2", "w3"];
let h_prefs: Prefs = vec![
    ("m1", vec!["w1", "w2", "w3"]),
    ("m2", vec!["w2", "w3", "w1"]),
    ("m3", vec!["w3", "w1", "w2"]),
];
let f_prefs: Prefs = vec![
    ("w1", vec!["m3", "m1", "m2"]),
    ("w2", vec!["m1", "m2", "m3"]),
    ("w3", vec!["m2", "m3", "m1"]),
];

let c_h: Prefs = vec![
    ("m1", vec!["w2", "w1", "w3"]),
    ("m2", vec!["w2", "w3", "w1"]),
    ("m3", vec!["w1", "w2", "w3"]),
];
let c_f: Prefs = vec![
    ("w1", vec!["m2", "m1", "m3"]),
    ("w2", vec!["m3", "m1", "m2"]),
    ("w3", vec!["m1", "m2", "m3"]),
];

// Exercice 3 : la declaration profitable reste a ecrire.
fn declaration_profitable_etudiant(_femme: &str, _homes_prefs: &Prefs, _femmes_prefs: &Prefs) -> Option<Vec<&'static str>> {
    // TODO etudiant : la declaration profitable, ou None s'il n'y en a pas
    None
}

/// La declaration donne-t-elle a `femme` un partenaire strictement meilleur ?
#[allow(dead_code)]
fn profitable(femme: &str, pp: &Prefs, rp: &Prefs, base: &str, declaration: &[&'static str]) -> bool {
    let mut declare = copie_prefs(rp);
    for e in declare.iter_mut() { if e.0 == femme { e.1 = declaration.to_vec(); } }
    let obtenu = gale_shapley(&keys(pp), pp, &keys(rp), &declare);
    let miroir = inverse(&obtenu);
    let partenaire = match_get(&miroir, femme).expect("apparie");
    rank_of(rp, femme, partenaire) < rank_of(rp, femme, base)
}

let reponses = vec![
    declaration_profitable_etudiant("w1", &copie_prefs(&h_prefs), &copie_prefs(&f_prefs)),
    declaration_profitable_etudiant("w2", &copie_prefs(&c_h), &copie_prefs(&c_f)),
];
if reponses.iter().all(|x| x.is_none()) {
    println!("Exercice 3 à compléter : déclaration gagnante sur C et absence de gain sur A");
} else {
    for &(femme, pp, rp, reponse) in [
        ("w1", &h_prefs, &f_prefs, &reponses[0]),
        ("w2", &c_h, &c_f, &reponses[1]),
    ].iter() {
        let honnete = gale_shapley(&keys(pp), pp, &keys(rp), rp);
        let base = match_get(&inverse(&honnete), femme).expect("apparie");
        let existence = permutations(&prefs_of(rp, femme).clone()).iter()
            .any(|d| profitable(femme, pp, rp, base, d));
        match reponse {
            None => assert!(!existence, "Ex3 gain manquant"),
            Some(r) => {
                let vraie_liste = prefs_of(rp, femme).clone();
                let meme_ensemble = r.len() == vraie_liste.len()
                    && r.iter().all(|x| vraie_liste.contains(x));
                assert!(meme_ensemble && profitable(femme, pp, rp, base, r),
                        "Ex3 déclaration sans gain");
            }
        }
    }
    println!("Exercice 3 : absence de gain et déclaration effectivement profitable validées");
};

Exercice 3 à compléter : déclaration gagnante sur C et absence de gain sur A


## Exercices — récapitulatif

Les trois exercices répartis dans le carnet font **reconstruire** l'algorithme et ses vérifications depuis zéro, pour que la non-stratégicité cesse d'être un résultat qu'on subit et devienne un mécanisme qu'on peut manipuler : implémenter la boucle de *deferred acceptance*, écrire un **vérificateur de stabilité** indépendant (qui ne suppose pas l'algorithme pour conclure), puis **chercher activement** un mensonge gagnant du côté répondant — l'expérience qui rend tangible l'asymétrie mesurée au résultat 3.


## À retenir

Gale–Shapley termine sur des listes strictes complètes de même taille et produit un appariage stable. Dans A, l’énumération des six bijections trouve deux résultats stables ; dans C, elle en trouve trois. La proposition des hommes ou des femmes sélectionne des extrêmes différents, comme le prédit l’optimalité du côté proposant.

Dans C, aucune permutation déclarée par un homme ne l’améliore lors de l’énumération de cette instance. Une recherche exhaustive trouve pour w2 le mensonge `[m3, m2, m1]` : elle passe de m1 (rang 2) à m3 (rang 1). w1 s’améliore aussi (rang 3→2), w3 reste au rang 2 ; m1 et m3 perdent un rang et m2 est inchangé. La stabilité sous préférences déclarées n’établit pas la stabilité selon les préférences vraies. Ces observations finies illustrent le mécanisme ; les théorèmes généraux d’optimalité/protection du proposant restent des résultats distincts.

Le NRMP existe depuis 1952 ; la variante où les résidents proposent est utilisée depuis 1998. Les échanges rénaux utilisent des cycles et dépassent le modèle biparti simple. Gibbard–Satterthwaite porte sur les mécanismes de choix social et n’est pas directement le théorème de manipulation de ce modèle.

**Exercices à compléter** : [implémenter l’acceptation différée](#exercice-1), [vérifier une paire bloquante](#exercice-2), [chercher une déclaration gagnante](#exercice-3). Les réponses restent des stubs. Dans le jumeau C#, des fixtures contrôlent leurs résultats et distinguent un stub d’un `null` légitime.


## Vérifiez votre compréhension

1. L’instance A porte deux appariages stables. L’algorithme de Gale–Shapley est-il non déterministe ?
2. L’énumération de C ne trouve aucun homme qui gagne en mentant. Est-ce le théorème général ?
3. Le mensonge de w2 donne-t-il aussi un gain aux autres femmes ?
4. L’issue est stable selon les listes déclarées. Est-elle nécessairement stable selon les préférences vraies ?

<details>
<summary>Éléments de réponse</summary>

1. Non. Avec un côté proposant et un ordre de traitement fixés, la procédure est déterministe ; elle sélectionne l’extrême stable optimal pour ce côté.
2. Non. La sortie est une vérification finie sur une seule instance et les permutations testées. L’optimalité du proposant est un théorème distinct du modèle d’appariage stable.
3. Dans cette instance, oui : w1 passe de rang 3 à 2. w3 reste au rang 2 ; m1 et m3 perdent un rang et m2 ne change pas. Ce relevé n’implique pas une loi générale de redistribution.
4. Non. Un certificat de stabilité utilise les listes déclarées à l’algorithme. La manipulation peut modifier ces listes et la mesure des rangs vrais est une question distincte.

</details>

## Pour aller plus loin

- **Gale & Shapley (1962)**, [*College Admissions and the Stability of Marriage*](https://doi.org/10.2307/2312726), référence fondatrice de l’acceptation différée.
- **NRMP**, [*How the Matching Algorithm Works*](https://www.nrmp.org/intro-to-the-match/how-matching-algorithm-works/) ; variante résidents-proposants utilisée depuis 1998, institution créée en 1952.
- [Version Python](GameTheory-16i-Appariage-Stable-Gale-Shapley-Python.ipynb), [Version C#](GameTheory-16i-Appariage-Stable-Gale-Shapley-CSharp.ipynb), [carnet 16 — Mécanismes](GameTheory-16-MechanismDesign-Python.ipynb), [GameTheory-16h](GameTheory-16h-Cooperation-LLM-Python.ipynb), [GameTheory-16j](GameTheory-16j-Screening-Loyer-Information-Python.ipynb).
